# 7.2 Распределения и выбросы — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 07 «Разведочный анализ» · страница 7.2.

В этом ноутбуке вы сравните гистограммы трат и проверите необычные записи. Четыре ячейки из раздела «Код» используют общую таблицу. Затем идут три самостоятельных примера о нулевых суммах, коде пропуска в возрасте и тратах при отсутствии заказов. Все частоты рассчитаны по строкам исходной выгрузки, включая повторы.

**Как работать.** Выполняйте ячейки по порядку сверху вниз.

Вывод получен на Python 3.13, NumPy 2.5, pandas 3.0, SciPy 1.18 и matplotlib 3.10.

## Где лежат данные

Распакуйте общий набор так, чтобы файл лежал по пути `data/eda/clients_export.csv`. Поместите папку урока рядом с папкой `data` и откройте ноутбук. Первая ячейка найдёт данные.

In [1]:
import os
import pathlib
import platform

import matplotlib
import numpy as np
import pandas as pd
import scipy

root = pathlib.Path.cwd()
while not (root / "data" / "eda").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/eda/
assert (root / "data" / "eda" / "clients_export.csv").exists(), \
    "Не найдена папка data/eda/. Скачайте общий набор темы 07 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

Python 3.13.13 | numpy 2.5.3 | pandas 3.0.5 | scipy 1.18.1 | matplotlib 3.10.9


## Сводные показатели и поиск необычных сумм

Первая ячейка читает выгрузку в таблицу `df`. Выполняйте следующие три ячейки по порядку.

### Среднее по выгрузке в 2,5 раза больше медианы

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv("data/eda/clients_export.csv")
num = df.select_dtypes("number").drop(columns="client_id")  # id — ярлык, а не величина
pct = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
print(num.describe(percentiles=pct).round(1).to_string())

s = df["spend_q1"]
print("\nусечённое среднее, по 10% с краёв:", round(stats.trim_mean(s, 0.1)))
print("MAD — медиана |x − медиана|:", stats.median_abs_deviation(s))

           age     tier  visits_q1  orders_q1    spend_q1
count  12040.0  12040.0    12040.0    12040.0     12040.0
mean      38.3      2.2        8.2        2.0     12379.2
std       14.8      1.2        3.3        1.5    259131.0
min       18.0      1.0        0.0       -1.0         0.0
1%        18.0      1.0        2.0       -1.0         0.0
5%        18.0      1.0        3.0        0.0         0.0
25%       29.0      1.0        6.0        1.0      2560.0
50%       37.0      2.0        8.0        2.0      5020.0
75%       44.0      3.0       10.0        3.0      9800.0
95%       59.0      5.0       14.0        5.0     25470.0
99%       99.0      5.0       18.0        6.0     52227.6
max       99.0      5.0       24.0       10.0  19931000.0

усечённое среднее, по 10% с краёв: 6080
MAD — медиана |x − медиана|: 3160.0


### В первом интервале гистограммы трат оказались почти все суммы

In [3]:
axes = num.hist(bins=30, figsize=(12, 6), layout=(2, 3))  # пять гистограмм одним вызовом

for ax in axes.flat[:num.shape[1]]:               # что нарисовано: высоты столбиков
    h = np.array([bar.get_height() for bar in ax.patches])
    lo = round(ax.patches[0].get_x(), 6) + 0.0   # + 0.0 убирает «−0»
    hi = ax.patches[-1].get_x() + ax.patches[-1].get_width()
    print(f"{ax.get_title():10} {lo:>4,.0f} … {hi:<11,.0f} пустых интервалов: {(h == 0).sum():>2}"
          f"   доля записей в самом высоком столбце: {h.max() / h.sum():>6.2%}")

age          18 … 99          пустых интервалов:  7   доля записей в самом высоком столбце: 10.47%
tier          1 … 5           пустых интервалов: 25   доля записей в самом высоком столбце: 40.00%
visits_q1     0 … 24          пустых интервалов:  5   доля записей в самом высоком столбце: 12.18%
orders_q1    -1 … 10          пустых интервалов: 18   доля записей в самом высоком столбце: 26.10%
spend_q1      0 … 19,931,000  пустых интервалов: 24   доля записей в самом высоком столбце: 99.96%


### Сумма 5 000 ₽ встречается в 44 раза чаще, чем в среднем соседние суммы

In [4]:
pos = s[(s > 0) & (s < 1_000_000)]                # строки с положительными тратами без пяти миллионных сумм
cnt = pos.value_counts().reindex(np.arange(10, 105_000, 10), fill_value=0)
near = (cnt.rolling(61, center=True, min_periods=1).sum() - cnt) / 60  # соседи ±300 ₽
ratio = (cnt / near).where((cnt.index % 500 == 0) & (cnt >= 10))  # круглые и не редкие

top = ratio.nlargest(4).index
print(pd.DataFrame({"записей": cnt[top], "среднее у соседей": near[top].round(1),
                    "во сколько раз": ratio[top].round(1)}).to_string())
round1000 = pos % 1000 == 0
print(f"Среди положительных сумм {round1000.mean():.1%} кратны 1 000 ₽.")
print(f"Кратны 1 000 ₽ и отличаются от 5 000 ₽: {(round1000 & (pos != 5000)).mean():.1%} всех положительных сумм в расчёте.")

          записей  среднее у соседей  во сколько раз
spend_q1                                            
5000          381                8.6            44.0
6500           13                6.5             2.0
6000           11                7.9             1.4
2500           15               12.0             1.2
Среди положительных сумм 4.6% кратны 1 000 ₽.
Кратны 1 000 ₽ и отличаются от 5 000 ₽: 0.9% всех положительных сумм в расчёте.


### Число сумм выше порога зависит от правила

In [5]:
q1, q3 = s.quantile([0.25, 0.75])
mad = stats.median_abs_deviation(s)
bounds = {"IQR": q3 + 1.5 * (q3 - q1),              # правило Тьюки
          "z > 3": s.mean() + 3 * s.std(),           # три стандартных отклонения
          "модиф. z > 3,5": s.median() + 3.5 * mad / 0.6745}  # модифицированная z-оценка
err = s > 1_000_000                                  # пять миллионных сумм, видимых на логарифмической шкале

rows = {rule: {"граница, ₽": round(b), "выше границы": (s > b).sum(),
               "из пяти миллионных сумм": (s[err] > b).sum()} for rule, b in bounds.items()}
print(pd.DataFrame(rows).T.to_string())

clean = s[~err]                                      # повторяем расчёт без пяти миллионных сумм
z_bound = clean.mean() + 3 * clean.std()
print(f"\nz > 3 без пяти миллионных сумм: граница {z_bound:,.0f} ₽, выше неё {(clean > z_bound).sum()} записей")
both = (s > bounds["IQR"]) & (s > bounds["модиф. z > 3,5"])
print("выше обеих границ, IQR и MAD:", both.sum(), "из", (s > bounds["IQR"]).sum())

                граница, ₽  выше границы  из пяти миллионных сумм
IQR                  20660           844                        5
z > 3               789772             5                        5
модиф. z > 3,5       21417           795                        5

z > 3 без пяти миллионных сумм: граница 37,329 ₽, выше неё 298 записей
выше обеих границ, IQR и MAD: 795 из 844


## Нюансы

В первой ячейке NumPy покажет два предупреждения при вычислении log10(0) и log1p(−1). Оба вызова вернут −inf.

### В 27 строках с нулевыми тратами число заказов неизвестно

In [6]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/eda/clients_export.csv")
s = df["spend_q1"]
zero = s == 0
print(f"нулей: {zero.sum()} из {len(s)} ({zero.mean():.1%})")
print("число заказов в строках с нулевыми тратами (−1 — пропуск):", df.loc[zero, "orders_q1"].value_counts().to_dict())
print("медиана: все", s.median(), "| строки с положительными тратами", s[~zero].median())

first = s.head(4).to_numpy()                  # 0, 0, 0 и 9 710 ₽
print(np.log10(first))
print(np.log1p(first).round(2))               # log(1 + x): ноль остаётся нулём
print(np.log1p(df["orders_q1"].to_numpy()).min())  # код пропуска −1 даёт −inf

нулей: 1717 из 12040 (14.3%)
число заказов в строках с нулевыми тратами (−1 — пропуск): {0.0: 1690, -1.0: 27}
медиана: все 5020.0 | строки с положительными тратами 6060.0
[      -inf       -inf       -inf 3.98721923]
[0.   0.   0.   9.18]
-inf


C:\Users\cosmi\AppData\Local\Temp\ipykernel_22504\46623185.py:12: RuntimeWarning: divide by zero encountered in log10
  print(np.log10(first))
C:\Users\cosmi\AppData\Local\Temp\ipykernel_22504\46623185.py:14: RuntimeWarning: divide by zero encountered in log1p
  print(np.log1p(df["orders_q1"].to_numpy()).min())  # код пропуска −1 даёт −inf


При вычислении log10(0) и log1p(−1) NumPy возвращает −inf и выводит предупреждение. Имя файла в тексте предупреждения зависит от среды запуска.

### После замены кода 99 на пропуск средний возраст снижается

In [7]:
import pandas as pd

df = pd.read_csv("data/eda/clients_export.csv")
age = df["age"]
vc = age.value_counts().sort_index()
print(vc.iloc[[0, 1, 2, -3, -2, -1]].to_string())   # края: 18, 19, 20 … 74, 75, 99
print(f"среднее {age.mean():.1f}, без 99 — {age[age != 99].mean():.1f}")

df["age_missing"] = age == 99              # 99 — код «не указан», то есть пропуск
df["age"] = age.mask(df["age_missing"])    # 99 → NaN, строка и другие столбцы остаются
print("пропусков:", df["age"].isna().sum(), "| медиана:", df["age"].median())

age
18.0    627
19.0    125
20.0    138
74.0      3
75.0      2
99.0    360
среднее 38.3, без 99 — 36.4
пропусков: 360 | медиана: 36.0


### В одной строке указаны траты при нуле заказов

In [8]:
import pandas as pd

df = pd.read_csv("data/eda/clients_export.csv")
d = df[df["spend_q1"] < 1_000_000]                   # исключаем пять миллионных сумм


def iqr_out(x):                                      # выход за границу по одному столбцу
    q1, q3 = x.quantile([0.25, 0.75])
    return (x < q1 - 1.5 * (q3 - q1)) | (x > q3 + 1.5 * (q3 - q1))


odd = d[(d["orders_q1"] == 0) & (d["spend_q1"] > 0)]  # траты без единого заказа
print(odd[["client_id", "visits_q1", "orders_q1", "spend_q1"]].to_string(index=False))
print("за границами IQR по spend_q1:", iqr_out(d["spend_q1"])[odd.index].any(),
      "| по orders_q1:", iqr_out(d["orders_q1"])[odd.index].any())
print("заказы есть, трат нет:", ((d["orders_q1"] > 0) & (d["spend_q1"] == 0)).sum())

per = d["spend_q1"] / d["orders_q1"].where(d["orders_q1"] > 0)  # траты на один заказ
print(f"Медиана трат на заказ: {per.median():,.0f} ₽. Число строк с тратами выше 20 000 ₽ на заказ: {(per > 20_000).sum()}")

 client_id  visits_q1  orders_q1  spend_q1
  101263.0          5        0.0    1000.0
за границами IQR по spend_q1: False | по orders_q1: False
заказы есть, трат нет: 0
Медиана трат на заказ: 2,983 ₽. Число строк с тратами выше 20 000 ₽ на заказ: 202
